# Kaggle Parallel Sweep Runner

Portable runner for the ablation matrix of
**"Do Register Tokens Regularize Vision Transformers Under Data Scarcity?"**

The full matrix is 12 runs (`K in {0, 1, 4, 8}` x seeds `{42, 1337, 3407}`). This
notebook executes an arbitrary **subset** of that matrix on a Kaggle GPU session so
several team members can run disjoint slices in parallel and merge the results back
into a single `outputs/` tree.

| Owner | Suggested slice | Runs |
| :--- | :--- | :--- |
| Session A | `REGISTERS = [0]` | EXP-01 .. EXP-03 |
| Session B | `REGISTERS = [1]` | EXP-04 .. EXP-06 |
| Session C | `REGISTERS = [4]` | EXP-07 .. EXP-09 |
| Session D | `REGISTERS = [8]` | EXP-10 .. EXP-12 |

Experiment indices are derived from the position in the **full** matrix, so a subset
session still writes canonically named directories (`outputs/exp07_k4_s42/`, ...) and
the aggregated results merge without renaming.

**Before running:** enable *Settings -> Accelerator -> GPU* and *Internet -> On*
(required for the CIFAR-100 download and the `timm` install).

## 1. Session diagnostics

In [ ]:
import platform
import subprocess

print("Python :", platform.python_version())
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                          "--format=csv,noheader"],
                         capture_output=True, text=True, check=True).stdout.strip())
except (FileNotFoundError, subprocess.CalledProcessError):
    print("No NVIDIA GPU detected - enable Settings -> Accelerator -> GPU before running the sweep.")

## 2. Obtain the project

Two supported sources:

* **Git clone** - set `REPO_URL` to the project remote (needs *Internet: On*).
* **Kaggle dataset** - upload the repository as a private dataset and set
  `DATASET_DIR` to its mount point, e.g. `/kaggle/input/aiac-res`.

The repository is copied into `/kaggle/working/aiac-res` so that `outputs/` and
`checkpoints/` are writable and persist in the session output.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = ""                       # e.g. "https://github.com/<org>/aiac-res.git"
DATASET_DIR = ""                    # e.g. "/kaggle/input/aiac-res"
PROJECT_ROOT = Path("/kaggle/working/aiac-res")

if not PROJECT_ROOT.exists():
    if DATASET_DIR and Path(DATASET_DIR).exists():
        source = Path(DATASET_DIR)
        # A dataset upload may nest the repository one directory deep.
        if not (source / "scripts" / "train.py").exists():
            candidates = [p for p in source.iterdir() if (p / "scripts" / "train.py").exists()]
            if candidates:
                source = candidates[0]
        shutil.copytree(source, PROJECT_ROOT)
    elif REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_ROOT)], check=True)
    else:
        raise SystemExit("Set REPO_URL or DATASET_DIR before running this cell.")

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())
print(sorted(p.name for p in Path.cwd().iterdir() if not p.name.startswith(".")))

## 3. Dependencies

In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED = {"timm": "timm>=0.9.12", "yaml": "pyyaml>=6.0.1"}
missing = [spec for module, spec in REQUIRED.items() if importlib.util.find_spec(module) is None]

if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

import timm
import torch

print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("timm :", timm.__version__)

## 4. Sweep slice configuration

`REGISTERS` and `SEEDS` select the subset of the matrix this session executes.
`NUM_WORKERS = 2` avoids the shared-memory dataloader deadlock that Kaggle
containers hit with four workers.

Set `EPOCHS = 1` once for a smoke pass before committing the session to the full
50-epoch schedule.

In [ ]:
REGISTERS = [4]                     # subset of [0, 1, 4, 8]
SEEDS = [42, 1337, 3407]            # subset of [42, 1337, 3407]
EPOCHS = None                       # None keeps the 50 epochs defined in the YAML configs
DATA_DIR = "/kaggle/working/data"   # CIFAR-100 download cache
NUM_WORKERS = 2                     # Kaggle-safe dataloader worker count

registers_arg = " ".join(str(k) for k in REGISTERS)
seeds_arg = " ".join(str(s) for s in SEEDS)
print(f"This session will run {len(REGISTERS) * len(SEEDS)} experiments: "
      f"K in [{registers_arg}] x seeds [{seeds_arg}]")

## 5. Preflight sanity check

In [ ]:
import subprocess
import sys

completed = subprocess.run([sys.executable, "scripts/preflight_check.py"], text=True)
if completed.returncode != 0:
    raise SystemExit("Preflight check failed - resolve the reported blocker before launching the sweep.")

## 6. Execute the sweep slice

`scripts/run_sweep.sh` handles per-run isolation, failure trapping into
`outputs/failures.log`, VRAM clearing between runs and the artifact contract check.
Output is streamed live so a long session can be monitored from the Kaggle log pane.

In [ ]:
import subprocess
import sys

command = [
    "bash", "scripts/run_sweep.sh",
    "--python", sys.executable,
    "--registers", registers_arg,
    "--seeds", seeds_arg,
    "--data-dir", DATA_DIR,
    "--extra-args", f"--num_workers {NUM_WORKERS}",
    "--skip-existing",
]
if EPOCHS is not None:
    command += ["--epochs", str(EPOCHS)]

process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="")
status = process.wait()

print(f"\nrun_sweep.sh exited with status {status}")
if status != 0:
    print("At least one run failed. Inspect outputs/failures.log below.")

In [ ]:
from pathlib import Path

failure_log = Path("outputs/failures.log")
print(failure_log.read_text() if failure_log.exists() else "No failures recorded.")

## 7. Aggregate this session's runs

The aggregator reduces whichever runs are present into `outputs/sweep_summary.json`.
Arms that this session did not execute are reported with `num_seeds = 0`; they are
filled in once the slices are merged.

In [ ]:
import sys

sys.path.insert(0, ".")
from src.utils.logger import aggregate_sweep_results, format_summary_table

summary = aggregate_sweep_results(outputs_dir="outputs/")
print(format_summary_table(summary))
print()
print("Completed runs:", summary["meta"]["completed_runs"], "/", summary["meta"]["expected_runs"])
print("Missing runs  :", ", ".join(summary["meta"]["missing_runs"]) or "none")

## 8. Package the results for merge-back

Only the per-run output directories are archived. Checkpoints are excluded by default
because 23 MB x 2 per run inflates the archive well past what is convenient to move
between sessions; set `INCLUDE_CHECKPOINTS = True` when the weights are needed for the
attention visualisations.

In [ ]:
import shutil
from pathlib import Path

INCLUDE_CHECKPOINTS = False
ARCHIVE_STEM = f"sweep_k{'-'.join(str(k) for k in REGISTERS)}"

staging = Path("/kaggle/working/_sweep_export")
if staging.exists():
    shutil.rmtree(staging)
(staging / "outputs").mkdir(parents=True)

for run_dir in sorted(Path("outputs").glob("exp*_k*_s*")):
    if run_dir.is_dir():
        shutil.copytree(run_dir, staging / "outputs" / run_dir.name)

if INCLUDE_CHECKPOINTS:
    (staging / "checkpoints").mkdir()
    for ckpt_dir in sorted(Path("checkpoints").glob("exp*_k*_s*")):
        if ckpt_dir.is_dir():
            shutil.copytree(ckpt_dir, staging / "checkpoints" / ckpt_dir.name)

archive = shutil.make_archive(f"/kaggle/working/{ARCHIVE_STEM}", "zip", staging)
print("Archive written to:", archive)
print("Size: %.1f MB" % (Path(archive).stat().st_size / 1024 ** 2))

## 9. Merging slices back into the main tree

1. Download `sweep_k*.zip` from the session output.
2. Unpack every slice into the project root; the `expXX_kY_sZ` directory names are
   globally unique, so the slices never collide.
3. Re-run the aggregator once over the merged tree to produce the final table input:

   ```bash
   python src/utils/logger.py --output_dir outputs/ --strict
   ```

   `--strict` returns a non-zero exit status while any of the 12 runs is still missing,
   which makes the merge verifiable rather than assumed.